# CeLLaTe Data processing

### Breakdown of the path where resulting **TSV files** come from & how they align to the Hugging Face dataset **OTAR3088/CeLLaTe_V2.0_with_vague**:
NB: The same logic is true for the equivalent 'no_vague' data walk

##### 1. Starting point
_'OTAR3088/CeLLaTe_V2.0_no_vague'_ on HuggingFace is the origin data source - the amalgamation of data originating from Pharmacological literature, single-cell immunocological literature and the CellFinder dataset.
- - - - - - - - - - - - - - - - - - - - - - - - - - - - 
##### 2. _CeLLaTe_V2.0_no_vague_ 
_'CeLLaTe_V2.0_no_vague'_ contains **10,267 rows** in total (when combining the train, validation, and test splits of the data) and includes four data sources: Single_Cell, CellFinder, Chembl_V2, and Chembl_V1.

- _'Chembl_V1'_ is an additional data source, introduced as an alteration to CeLLaTe, relating to transformer model fine-tuning
- This is considered out of scope for the final clean dataset; on removal CeLLaTe is **6,956 rows** long.
- - - - - - - - - - - - - - - - - - - - - - - - - - - - 
##### 3. _cellate_final.tsv_ 
_'cellate_final.tsv'_ is produced in this notebook, and provides a sanity-checked and cleaned CeLLaTe representation. Checks / changes here are illustrated by the user story defined below.

- - - - - - - - - - - - - - - - - - - - - - - - - - - - 
##### 4. _cellate_final_with_pmcid.tsv_
_'cellate_final_with_pmcid.tsv'_ is created in the script **map_sentences_to_pmcids.py**
- _'cellate_final.tsv'_ is taken as an input
- Raw text data from local source files (output annotation files from LabelStudio / CellFinder) are also read-in
- Script performs normalized substring matching on the sentence text to find matching PubMed Central IDs (PMCID)
- Script outputs the result with the matched PMCIDs in the first column - this is _'cellate_final_with_pmcid.tsv'_
- Other than replacing the index column Unnamed: 0 with the mapped PMCID column, the actual content of the sentence, entities, and data_source columns is identical row-for-row to _'cellate_final.tsv'_.
- Both files contain exactly **6,956 rows**.

*************************
The 6,956 rows in the data represent 6,956 annotated sentence instances.

There are 37 duplicate sentences, which occur due to the fact that there are repeated sentences present in some source papers.

See /Users/withers/GitProjects/OTAR3088/Data_mining/CeLLaTe/duplicate_sentences.tsv for more detail.

# User story:
_As an end user of the CeLLaTe dataset, I wish to have an **un-edited**, **whole** version of CeLLaTe. There should be no splits, and no entity tags outside of those from the base dataset schema - **'CellType', 'CellLine' and 'Tissue'**. The data will originate from the sources **'ChEMBL assay descriptions', 'Single Cell' and 'CellFinder'** to ensure all have been annotated to the standard of the annotation guidelines described by the project curators. This will ensure to me that I am accessing a gold-standard dataset._

### Final dataset to be formatted as follows:
✅ The dataset 'OTAR3088/CeLLaTe_V2.0_no_vague', which has been processed into test, train, eval splits will be read in from Hugging Face.

✅ Test, train and eval splits will be merged.

✅ ChEMBL data, single cell data and CellFinder will be the only sources in this set.

✅ It will be sanity checked that there are no historic annotations outside of the types 'CellType', 'CellLine' and 'Tissue'.

✅ Cleaned output will be written to the OTAR3088 Hugging Face space, under the name 'CeLLaTe_all'

- data_source names will be cleaned to best describe data sources

In [1]:
# Data read in and train / test / val sets merged

from datasets import load_dataset
import pandas as pd

# dataset = load_dataset("OTAR3088/CeLLaTe_V2.0_no_vague")
dataset = load_dataset("OTAR3088/CeLLaTe_V2.0_with_vague")
train = dataset["train"].to_pandas()
val = dataset["validation"].to_pandas()
test = dataset["test"].to_pandas()

res = pd.concat([train, val, test], axis=0, ignore_index=True)
# res.to_csv('./cellate_all_messy.tsv', sep='\t')
res

,sentence,entities,data_source
0,A single-cell and spatial genomics atlas of hu...,"[{'end': 66, 'label': 'CellType', 'start': 50,...",Single_Cell
1,Fibroblasts sculpt the architecture and cellul...,"[{'end': 11, 'label': 'CellType', 'start': 0, ...",Single_Cell
2,Here we constructed a spatially resolved atlas...,"[{'end': 90, 'label': 'Tissue', 'start': 86, '...",Single_Cell
3,We define six major skin fibroblast subtypes i...,"[{'end': 44, 'label': 'CellType', 'start': 20,...",Single_Cell
4,We characterize two fibroblast subtypes furthe...,"[{'end': 39, 'label': 'CellType', 'start': 20,...",Single_Cell
...,...,...,...
10285,"Wenjun Chai: Resources, Methodology, Formal an...",[],Chembl_V1
10286,"Ke Xue: Software, Data curation.",[],Chembl_V1
10287,"Hongyu Pan: Writing – review & editing, Superv...",[],Chembl_V1
10288,"Mingxia Yan: Writing – review & editing, Super...",[],Chembl_V1


In [2]:
# Check 1: Are there any sources extra to final, clean CeLLaTe?

screen_df = res[~res['data_source'].isin(['Single_Cell', 'Chembl_V2', 'CellFinder'])]
extra = set(screen_df['data_source'].to_list())

print(f'The following are out of scope data sources for final dataset: {extra}')

The following are out of scope data sources for final dataset: {'Chembl_V1'}


In [3]:
# Remove extra sources

res = res[res['data_source'].isin(['Single_Cell', 'Chembl_V2', 'CellFinder'])]

In [4]:
# Check 2: Are there any erronous annotation types?
import re

labels = set()
for val in res['entities']:
    # List of dict objects
    if len(val) > 0:
        for entity in val:
            if isinstance(entity, dict) and 'label' in entity:
                labels.add(entity['label'])

# Are any labels erronous
allowed = {'CellType', 'CellLine', 'Tissue'}
invalid = labels - allowed
print("All unique labels found:", sorted(list(labels)))
print("Invalid labels:", sorted(list(invalid)))

All unique labels found: ['CellLine', 'CellType', 'Tissue']
Invalid labels: []


In [6]:
# Write CeLLaTe final to Hugging Face
from datasets import Dataset

dataset = Dataset.from_pandas(res, preserve_index=False)
dataset.push_to_hub(
    "OTAR3088/CeLLaTe_all_with_vague"
)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/7 [00:00<?, ?ba/s]

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

                                        :  21%|##        |  158kB /  765kB            

CommitInfo(commit_url='https://huggingface.co/datasets/OTAR3088/CeLLaTe_all_with_vague/commit/81df95bb7e14d79f207c26211aa490efc2a40b56', commit_message='Upload dataset', commit_description='', oid='81df95bb7e14d79f207c26211aa490efc2a40b56', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/OTAR3088/CeLLaTe_all_with_vague', endpoint='https://huggingface.co', repo_type='dataset', repo_id='OTAR3088/CeLLaTe_all_with_vague'), pr_revision=None, pr_num=None)

In [7]:
res.to_csv('./cellate_final_with_vague.tsv', sep='\t')